In [1]:
#Convolution Neural Network using PyTorch for MNIST Dataset

#Import packages
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import torchvision.datasets as dsets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
#Setup the CNN Sub class for MNIST Dataset

class CNNNeuralNet(nn.Module):

    def __init__(self, in_size=1, out1_size=64, out2_size=128):
        super().__init__()

        self.cnn1 = nn.Conv2d(in_channels=in_size,out_channels=out1_size,kernel_size=3,padding=1) # 64x28x28
        self.maxpool1 = nn.MaxPool2d(kernel_size=2,stride=2) # stride=2 reduces dim by half 64x14x14

        self.cnn2 = nn.Conv2d(in_channels=out1_size,out_channels=out2_size,kernel_size=3,stride=1,padding=1) # 128x14x14
        self.maxpool2 = nn.MaxPool2d(kernel_size=2,stride=2) # 128x7x7

        self.flatten  = nn.Flatten()

        self.fc1 =  nn.Linear(out2_size*7*7,64) # 128*7*7 -> 64
        self.fc2 =  nn.Linear(64,32) # 64 -> 32
        self.fc3 =  nn.Linear(32,10) # 32 -> 10 ( 10 output channels in MNIST)

        

    def forward(self,x):
        
        x = torch.relu(self.cnn1(x))
        x = self.maxpool1(x)
    
        x = torch.relu(self.cnn2(x))
        x = self.maxpool2(x)

        x = self.flatten(x)

        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [3]:
# setup the dataset

train_dataset = dsets.MNIST(root = './Data',download=True,train=True,transform=transforms.ToTensor())
test_dataset = dsets.MNIST(root = './Data',download=True,train=False,transform=transforms.ToTensor())

train_loader = DataLoader(dataset=train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(dataset=test_dataset,batch_size=128,shuffle=False)

In [5]:
#Setup model training parameters

model = CNNNeuralNet()

print(f"Is CUDA available: {torch.cuda.is_available()}")

model.to('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Model architecture: {model.state_dict()}")

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

epochs = 5

Is CUDA available: False
Model architecture: OrderedDict({'cnn1.weight': tensor([[[[-0.0299, -0.3304, -0.1184],
          [ 0.0911,  0.1583, -0.1444],
          [-0.2407,  0.3081, -0.2424]]],


        [[[-0.2665, -0.2811, -0.2093],
          [ 0.0377, -0.1167,  0.3207],
          [ 0.0361, -0.3216, -0.1891]]],


        [[[ 0.1987,  0.2094, -0.1346],
          [-0.1604, -0.0581, -0.0884],
          [ 0.1472, -0.1482, -0.0706]]],


        [[[ 0.2927,  0.2160,  0.1774],
          [ 0.0503, -0.2420,  0.2207],
          [ 0.1097, -0.1462, -0.1762]]],


        [[[ 0.1929,  0.2405,  0.2394],
          [-0.3147,  0.0710,  0.1543],
          [-0.2842, -0.1697,  0.0872]]],


        [[[ 0.2290,  0.2883,  0.2507],
          [ 0.1419,  0.1398, -0.2612],
          [-0.1101, -0.1867, -0.0400]]],


        [[[-0.2076,  0.0510,  0.1665],
          [ 0.2454,  0.1580, -0.1132],
          [ 0.0756, -0.1543, -0.0900]]],


        [[[-0.1219,  0.2263, -0.1002],
          [-0.0356, -0.2064,  0.0076],
  

In [ ]:
#Train the model - setup training loop

model_metrics = { 'Training_Loss':[], 'Validation_Accuracy': []}

def train_model(n_epochs):
    for epoch in range(n_epochs):
        running_loss = 0
        model.train()
        for x,y in train_loader:
            x,y = x.to('cuda' if torch.cuda.is_available() else 'cpu'), y.to('cuda' if torch.cuda.is_available() else 'cpu')
            optimizer.zero_grad()
            yhat = model(x)
            loss = criterion(yhat,y)
            running_loss += loss.item() 
            loss.backward()
            optimizer.step()
        training_loss = running_loss / len(train_loader)
        model_metrics['Training_Loss'].append(training_loss)

        model.eval()
        with torch.no_grad():
            correct = 0
            for x_test,y_test in test_loader:
                z = model(x_test)
                _,yhat = torch.max(z, dim=1)
                correct += (yhat==y_test).sum().item()
            validation_accuracy = correct / len(test_loader.dataset)
            model_metrics['Validation_Accuracy'].append(validation_accuracy)
        
        #if(epoch+1) % 5 == 0:
        print(f"Training Loss: {model_metrics['Training_Loss'][-1]: .4f}",flush=True)
        print(f"Validation Accuracy: {model_metrics['Validation_Accuracy'][-1]: .4f}",flush=True)

#Call the training function
train_model(epochs)

In [ ]:
# Plot the training loss and validation accuracy

_,(ax1,ax2) =  plt.subplots(1,2,figsize=(12,6))

ax1.plot(range(1, epochs+1), np.array(model_metrics['Training_Loss']), marker='o', color='r', ls=':', label='Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('CNN Training Loss Plot')
ax1.legend()
ax1.grid()

ax2.plot(range(1, epochs+1),np.array(model_metrics['Validation_Accuracy']), marker = 'o',color = 'g', ls='--',label='Validation accuracy per epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Accuracy')
ax2.set_title('CNN Validation Accuracy Plot')
ax2.legend()
ax2.grid()

plt.tight_layout()

plt.show()


In [ ]:
#Plot the actual and predicted labels along the orginal image
model.eval()
fig, axes = plt.subplots(2, 5, figsize=(18, 6)) 

axes = axes.flatten()
with torch.no_grad():
    for i, (x, y) in zip(range(10), test_loader):
        z = model(x)
        _, yhat = torch.max(z, 1)
        
        axes[i].imshow(x[i].reshape(28,28), cmap='gray') # single image [28,28]
        axes[i].set_title(f'Actual: {y[i].item()} | Predicted: {yhat[i].item()}')
        axes[i].axis('off')
plt.tight_layout(pad=1.0)
plt.show()